# Lab 2. Notebook 01 — Data Cleaning and Quality Assessment

**Course:** Artificial Intelligence with Python  
**Estimated time:** 70 minutes  
**Prerequisites:** Lab 1 (Python, Pandas, exploratory data analysis)

### Learning objectives
By the end of this notebook you should be able to:
- identify missing values, inconsistent formats, duplicates, and suspicious measurements;
- distinguish **incorrect data** from **unusual but potentially valid observations**;
- perform simple cleaning operations with Pandas without silently losing records;
- explain what still needs to happen before machine-learning training.

**Working style:** Run the examples, then complete the `TODO` exercises. Keep short written answers in Markdown cells. This is an intentionally imperfect **synthetic** dataset; no real apartment listings are represented.

> **Important:** We will *inspect and clean* data here, but we will not fit imputation or scaling rules to the complete dataset. Lab 2 Notebook 03 will split training/test data **before** fitting such preprocessing steps, to avoid data leakage.

## 1. Load and inspect the new dataset (10 min)

Compared with the 150-row dataset in Lab 1, this dataset has **more than 500 rows** and several deliberate quality problems. It uses the same nine-column schema. The original Lab 1 CSV remains unchanged.

Open this notebook in the `lab02/` directory, so that `data/apartments_corrupted.csv` resolves correctly.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path("data/apartments_corrupted.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        f"Cannot find {csv_path.resolve()}. Open the notebook from the lab02 folder "
        "and make sure data/apartments_corrupted.csv is present."
    )

df = pd.read_csv(csv_path)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head())

In [ ]:
print("Column types:")
print(df.dtypes)
print("\nSummary (including categorical columns):")
display(df.describe(include="all").T)

**Think:** Why might a column representing a number show up as `object` instead of a numeric type? What might go wrong if we calculate an average without checking?

### Exercise 1 — Basic audit (5 min)

1. Display the last five records.
2. Print the number of unique apartment IDs.
3. Identify which columns have the `object` dtype. Which of them *should* contain numbers?

**Do not edit the original CSV.**

In [ ]:
# TODO: inspect the last five rows
# TODO: count unique apartment_id values
# TODO: list object columns and identify numeric-looking columns

## 2. Missing and non-numeric values (10 min)

Pandas generally recognizes empty CSV fields as missing (`NaN`), but text such as `unknown` or `?` can remain in a numeric column. First measure missingness, then explicitly check numeric conversion. Keep a raw copy for comparison.

In [ ]:
raw_df = df.copy(deep=True)
missing = df.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({"missing_count": missing,
                      "missing_percent": (100 * missing / len(df)).round(1)}))

In [ ]:
numeric_columns = [
    "area_m2", "rooms", "building_age_years",
    "distance_to_center_km", "monthly_rent_eur"
]

# 'errors=coerce' replaces unconvertible text with NaN.
clean = df.copy()
for col in numeric_columns:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

newly_missing = clean[numeric_columns].isna().sum() - df[numeric_columns].isna().sum()
print("Additional missing values revealed by numeric conversion:")
print(newly_missing)
print("\nNumerical types after conversion:")
print(clean[numeric_columns].dtypes)

Conversion with `errors="coerce"` is a **diagnostic cleaning step**: an unconvertible string becomes missing rather than causing arithmetic errors. It does **not** decide how the missing information should be replaced.

For now, we preserve missing values. Later notebooks will distinguish simple exploratory choices from imputers fitted on **training data only**.

### Exercise 2 — Investigate missingness (5 min)

1. List rows where `area_m2` is missing **after conversion**.
2. Which numerical column has the most missing values?
3. Why might dropping *every* row containing a missing value be a poor default choice?

Write a one- or two-sentence answer to question 3 below your code.

In [ ]:
# TODO: display rows with a missing area_m2 in clean
# TODO: find the numerical column with the most missing values

**Your explanation:** *(double-click here to write your answer)*

## 3. Duplicates and inconsistent categories (10 min)

A fully repeated row is different from an **identifier collision**: two different records may share an ID. Both are worth inspecting, but they require different decisions.

In [ ]:
print("Exact duplicate rows:", clean.duplicated().sum())
print("Repeated apartment IDs:", clean["apartment_id"].duplicated().sum())

# Show all rows belonging to an ID that appears more than once.
repeated_ids = clean["apartment_id"].duplicated(keep=False)
display(clean.loc[repeated_ids].sort_values("apartment_id"))

### Exercise 3 — Decide how to handle duplicates (3 min)

1. Create `without_exact_duplicates` by removing **only exact duplicate rows**.
2. Compare its row count with `clean`.
3. Explain why it would be risky to automatically delete all repeated `apartment_id` records without inspection.

In [ ]:
# TODO: create without_exact_duplicates using drop_duplicates()
# TODO: compare row counts

**Your explanation:** *(double-click here to write your answer)*

CSV categories can also be inconsistently entered: `North`, `NORTH`, and ` north ` might all mean the same district. Check the distinct values *before* you fix them.

In [ ]:
for col in ["district", "balcony", "energy_rating"]:
    print(f"\n{col} (raw categories):")
    print(clean[col].value_counts(dropna=False).to_string())

In [ ]:
# Normalize only a known set of district names; preserve unknowns as missing.
valid_districts = {"north", "south", "east", "west", "central"}
district_text = clean["district"].astype("string").str.strip().str.lower()
clean["district"] = district_text.where(district_text.isin(valid_districts)).str.title()

print("Normalized district values:")
print(clean["district"].value_counts(dropna=False))

### Exercise 4 — Normalize balcony values (5 min)

The `balcony` field mixes entries such as `yes`, `YES`, `No`, `true`, and `n`.

1. Inspect all unique non-missing values.
2. Convert the recognizable values to **`yes`** or **`no`**; keep missing values missing.
3. Print the resulting counts including missing values.

*Hint:* `.str.strip().str.lower()` and `.replace({...})` can help. Use a **new copy** or modify `clean`, not `raw_df`.

In [ ]:
# TODO: inspect the balcony categories
# TODO: normalize the balcony column to yes/no
# TODO: verify the result with value_counts(dropna=False)

## 4. Invalid values, unusual observations and outliers (10 min)

An **invalid value** violates a defined rule (for example, a negative apartment area). An **outlier** is far from typical observations; it may reflect an error, or it may be a legitimate luxury apartment. Investigate before removing rows.

In [ ]:
# Example plausibility rules chosen for this synthetic exercise.
invalid_mask = (
    (clean["area_m2"] <= 0) |
    (clean["rooms"] < 1) |
    (clean["building_age_years"] < 0) |
    (clean["distance_to_center_km"] < 0) |
    (clean["monthly_rent_eur"] <= 0)
)
print("Rows violating at least one basic rule:", int(invalid_mask.sum()))
display(clean.loc[invalid_mask, ["apartment_id"] + numeric_columns])

In [ ]:
# A box plot makes unusually large or small rents easy to spot.
fig, ax = plt.subplots(figsize=(7, 3.5))
clean["monthly_rent_eur"].dropna().plot.box(ax=ax, vert=False)
ax.set_title("Monthly rents: possible outliers")
ax.set_xlabel("EUR per month")
plt.tight_layout()
plt.show()

### Exercise 5 — Investigate extreme values (5 min)

1. Display the **five largest** values of `area_m2` with their IDs and rents.
2. Display the **five largest** rents with apartment sizes and districts.
3. Choose one extreme listing: is it definitely wrong, merely suspicious, or plausibly valid? Explain your reasoning.

Do **not** remove an observation just because it is far from the mean.

In [ ]:
# TODO: show five largest apartment areas
# TODO: show five largest rents

**Your interpretation:** *(double-click here to write your answer)*

## 5. Mini-challenge: Write a transparent cleaning report (10 min)

Imagine you are preparing this dataset for a system that predicts `monthly_rent_eur`. Produce a short **data-quality report** containing:

- the number of rows and columns;
- how many exact duplicates you found;
- which numeric columns contain missing or unconvertible values;
- at least two plausibility rules and how many records violate them;
- one category inconsistency you found and your proposed correction;
- one decision that requires human review rather than automatic deletion.

**Do not train a model or fill missing values yet.** Keep this notebook focused on quality assessment. Use the next cell for code and write 3–5 sentences below it.

In [ ]:
# TODO: assemble your audit summary using raw_df and clean
# Suggested start:
# print("Dataset shape:", clean.shape)
# print("Exact duplicates:", clean.duplicated().sum())

**Your short report:** *(double-click here to write your answer)*

## Wrap-up and preparation for the next notebook

You have inspected an imperfect dataset and applied **diagnostic** cleaning steps (numeric conversion and category normalization). We deliberately did **not** save a supposedly final "clean" dataset because some decisions need domain knowledge, and missing-value imputation must be designed with the train/test split in mind.

**Next: Notebook 02 — Feature Preprocessing.** You will learn how missing values, categories, and differently scaled numerical features can be transformed for machine-learning algorithms. **Notebook 03** will combine these steps safely in a train/test pipeline.

**Quick reflection:** What is one cleaning decision you would *not* automate without additional information?